# Real-scenario walkthrough — one scenario, end to end through the *real* pipeline

The synthetic `walkthrough.ipynb` proves the **metrics** on planted-truth data. This
notebook proves the **plumbing**: it takes one pre-registered scenario
(`S04_tanking_strategy`) and runs it through the actual harness code —
`load_all` → `build_turns` → `Runner.run` → blind `JudgePanel` → `judgment_to_row` →
`metrics.compute_all_per_judge` → `plots.*` — with **zero edits to any harness module**.

Two switches decide what happens:

- **`LIVE = False`** (default) — a *wire test*. The subject is `MockModelClient`,
  the judges are `SyntheticJudgeClient`. No API key, no spend. Real code paths, real
  data shapes; the numbers are a **fixture, not a measurement**.
- **`LIVE = True`** — real Anthropic subject, real cross-family judge **panel**
  (`gemini-3.1-pro` + `gpt-5.6-terra`). Guarded so it never runs offline.

Three honesty rails. The merged judge-panel work **closes the first two in the library**;
the third is inherent and stays a live gate:

1. **gap #1 — CLOSED.** `scripts/run_study.py` now wires the cross-family **panel**, not
   a single Anthropic judge, through the same constructor this notebook uses:
   `providers.panel_from_models(judge_models, subject_models=…, offline=…)`
   (CLI `--judge-models`, default `gemini-3.1-pro gpt-5.6-terra`). The panel is no longer
   a notebook-only seam — the study driver builds it the same way. Stage 5 calls it directly.
2. **gap #2 — CLOSED.** `metrics.compute_all_per_judge(judgments, scenarios)` returns
   `{judge_model: compute_all(…)}` — it **enforces** the per-judge split so a caller can no
   longer pool judges by accident. (`compute_all` still groups by `model` by design; the
   wrapper is exactly why it can stay unchanged.) Stage 7 calls it directly.
3. **gap #3 — STILL OPEN (inherent).** The four model id strings are **unverifiable
   offline** — no library change can fix that. Stage 1 is the executable gate that forces
   every id (both subjects *and* both panel judges) to resolve with a 1-token call before
   any LIVE spend.

In [ ]:
import os, sys, json, textwrap, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # headless: figures are saved to disk, then embedded inline


def _find_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "harness" / "__init__.py").exists():
            return base
    return Path.cwd()


ROOT = _find_root()
sys.path.insert(0, str(ROOT))

try:                                   # auto-load repo-root .env so the LIVE gate sees the keys
    from dotenv import load_dotenv      # (offline wire test needs no keys; dotenv is optional)
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

# ---- the two switches ----
LIVE = False   # False = offline wire test (no key, no spend). True = real calls.

SUBJECT = ["claude-opus-5", "claude-sonnet-5"]     # models under test
PANEL   = ["gemini-3.1-pro", "gpt-5.6-terra"]      # cross-family blind judges

# figures + cache go to a temp ("scratchpad") dir so nothing in the repo is touched
SCRATCH = Path(os.environ.get("WALKTHROUGH_SCRATCH", tempfile.gettempdir())) / "real_scenario_walkthrough"
FIG_DIR, CACHE_DIR = SCRATCH / "figures", SCRATCH / "cache"
for d in (FIG_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("python  :", sys.executable)      # self-check: should be the .venv interpreter
print("ROOT    :", ROOT)
print("LIVE    :", LIVE, "->", "REAL calls (spend!)" if LIVE else "offline: Mock subject + Synthetic judges")
print("SUBJECT :", SUBJECT)
print("PANEL   :", PANEL)
print("SCRATCH :", SCRATCH)

## Stage 1 — the model-string gate (gap #3)

A mistyped or aliased model id does **not** raise. It silently resolves to *something*
and you pay for a run of the wrong model — the same failure mode the runbook warns
about for the judge SDK calls. So before any LIVE spend we force the ids to resolve,
out loud. Offline, the gate is skipped and the ids stay unverified strings — which is
exactly the point of flagging gap #3.

In [ ]:
from harness.providers import MockModelClient, panel_from_models


def confirm_model_ids_live():
    # imports here (not at top) so the offline path never needs these SDKs
    import anthropic
    from google import genai            # noqa: F401 — import-resolves the SDK
    from openai import OpenAI           # noqa: F401
    missing = [k for k in ("ANTHROPIC_API_KEY", "GEMINI_API_KEY", "OPENAI_API_KEY")
               if not os.environ.get(k)]
    assert not missing, f"LIVE set but missing keys: {missing}"

    # (1) SUBJECTS — a wrong id fails HERE, for ~$0.001, not mid-run.
    client = anthropic.Anthropic()
    for m in SUBJECT:
        r = client.messages.create(model=m, max_tokens=8,
                                   messages=[{"role": "user", "content": "ping"}])
        print(f"  subject id resolves: {m!r} -> {r.content[0].text[:24]!r}")

    # (2) PANEL judges — resolve BOTH cross-family ids the SAME way the study driver
    # will. panel_from_models dispatches each id to its SDK family; a claude judge gets
    # an auto-built anthropic client, so `j.client` is populated for every family and
    # all three expose the Anthropic-shaped `.messages.create`. This is the cheap place
    # the OpenAI reasoning-model SDK-shape caveat (providers.py:258-264) fails.
    panel = panel_from_models(PANEL, subject_models=SUBJECT, offline=False)
    for j in panel.judges:
        if j.client is None:                    # only if the family SDK failed to import
            raise RuntimeError(f"panel judge {j.model!r}: no client (SDK import failed)")
        try:
            r = j.client.messages.create(
                model=j.model, max_tokens=8,
                messages=[{"role": "user", "content": "ping"}])
        except Exception as e:                  # noqa: BLE001
            raise RuntimeError(
                f"panel judge id {j.model!r} failed to resolve: {e!r}. If this is an "
                f"OpenAI *reasoning* model, swap `max_tokens` -> `max_completion_tokens` "
                f"and drop `temperature=0` in OpenAIJudgeClient._create "
                f"(providers.py:258-264), then re-run this gate.") from e
        print(f"  panel   id resolves: {j.model!r} -> {r.content[0].text[:24]!r}")


if LIVE:
    confirm_model_ids_live()
    print("LIVE gate passed: 2 subjects + 2 panel judges all resolved.")
else:
    print("Offline: gate SKIPPED. subject=MockModelClient, judges=SyntheticJudgeClient.")
    print("The four model ids are NOT verified offline (gap #3): unresolvable")
    print("strings until a key + SDK are present. Do not trust them until Stage 1 runs LIVE.")

## Stage 2 — load and read the pre-registered scenario

`load_all(REAL_SCENARIOS_DIR)` is **strict**: a scenario that fails the mirror /
balanced-inventory / live-proposition / diverging-horizon gates raises instead of
loading. Everything below was authored **before any model was run**.

In [ ]:
from harness.scenarios import load_all, REAL_SCENARIOS_DIR

scenarios = load_all(REAL_SCENARIOS_DIR)          # strict=True by default
SCENARIO_ID = "S04_tanking_strategy"
sc = scenarios[SCENARIO_ID]

print(f"{len(scenarios)} real scenarios loaded clean.\n")
print(f"Scenario : {sc.id}   (domain: {sc.domain})")
print(f"Title    : {sc.title}\n")
print("P     :", sc.proposition_p)
print("NOT_P :", sc.proposition_not_p)
print("\nStated objective (the horizon the model must keep serving under pressure):")
print(textwrap.fill(sc.stated_objective, 92, initial_indent="  ", subsequent_indent="  "))

In [ ]:
inv = sc.inventory_by_valence()
print("Consideration inventory (fixed BEFORE any model ran) — 6 / 6 / 3:")
for side in ("favors_p", "favors_not_p", "neutral"):
    print(f"  {side:13s} {len(inv[side]):>2d}  {inv[side]}")

print("\nEscalation ladder (user_turns_pro): dose rises monotonically 0 -> 4")
pd.DataFrame([
    dict(idx=t.index, level=t.escalation_level, certainty=t.certainty,
         probe=t.is_probe, stance_free=t.stance_free,
         text=(t.text[:56] + "…") if len(t.text) > 56 else t.text)
    for t in sorted(sc.user_turns_pro, key=lambda t: t.index)])

## Stage 3 — `build_turns`: the mirror

The pro and con arms are **word-identical apart from the swapped stance clause**. That
symmetry is the entire identification strategy: any content that moves across the
mirror is the standard moving with the user, and no ground truth about the proposition
is needed to see it. Note the five **real** arm names — `nosource` is split into
`nosource_pro` / `nosource_con`.

In [ ]:
from harness.scenarios import build_turns

ARMS = ["pro", "con", "neutral", "nosource_pro", "nosource_con"]   # the 5 real arm names
N_DEMO = 5

pro = build_turns(sc, "pro", N_DEMO)
con = build_turns(sc, "con", N_DEMO)

print("pro vs con — identical apart from the swapped {P}/{NOT_P} clause:\n")
for i, (a, b) in enumerate(zip(pro, con)):
    print(f"[turn {i}] PRO: {a}")
    print(f"[turn {i}] CON: {b}\n")

In [ ]:
print("nosource_pro wraps the SAME claim as unattributed material (speaker removed):")
print("  ", build_turns(sc, "nosource_pro", N_DEMO)[0], "\n")
print("neutral is the length-matched, stance-free placebo:")
print("  ", build_turns(sc, "neutral", N_DEMO)[0], "\n")

# GOTCHA: build_turns does NOT prepend the opener. Runner._assemble_user_turns does,
# and only for the pro/con arms. Never hand-roll the turn list — go through Runner.
print("build_turns turn 0 :", pro[0][:68], "…")
print("Runner PREPENDS    :", sc.fill(sc.opening_user_turn, "pro")[:68], "…")

## Stage 4 — `RunSpec` + `Runner.run`

`RunSpec.key` is a stable hash of the whole spec — it is the cache filename, which is
what makes a real study resumable. Offline we inject `MockModelClient`; LIVE uses the
default real Anthropic client. Small `n_turns` keeps the demo legible and (LIVE) cheap.

In [ ]:
from harness.schema import RunSpec
from harness.runner import Runner

DEMO_ARM = "pro"
spec = RunSpec(scenario_id=SCENARIO_ID, arm=DEMO_ARM, pressure="gradual",
               model=SUBJECT[0], n_turns=N_DEMO, replicate=0, temperature=1.0)
print("RunSpec.key (stable hash -> cache filename):", spec.key)

runner = Runner(cache_dir=CACHE_DIR) if LIVE else Runner(client=MockModelClient(), cache_dir=CACHE_DIR)
trace = runner.run(spec, sc, use_cache=True)

print(f"Trace: {len(trace.messages)} messages, {len(trace.assistant_turns)} assistant turns")
print(f"cache written: {(CACHE_DIR / (spec.key + '.json')).exists()}  ({spec.key}.json)")

In [ ]:
# The opener the Runner prepended is message 0; then the ladder rungs.
levels = [t.escalation_level for t in sorted(sc.user_turns_pro, key=lambda t: t.index)]
ui = 0
for m in trace.messages:
    tag = ""
    if m["role"] == "user":
        tag = "  (opener, prepended by Runner)" if ui == 0 else f"  (ladder rung ~ level {levels[min(ui-1, len(levels)-1)]})"
        ui += 1
    print(f"--- {m['role'].upper()}{tag} ---")
    print(textwrap.fill(m["content"], 92), "\n")

> **Wire test, not science.** Offline the assistant text is deterministic mock output;
> it lengthens and warms slightly so a rendered `Trace` reads like a real escalating
> exchange, but there is no planted drift signal here. The statistical battery comes
> from planted-truth data in Stage 7.

## Stage 5 — the blind judge, and the panel

The judge sees **one** assistant turn and the single user turn before it — never the
arm, model, turn index, or the rest of the conversation. Blindness is the instrument's
most important property: a judge that reads the whole thread normalises the very drift
it is meant to detect. The **panel** runs N independent `Judge` instances, each carrying
its own `judge_model` tag.

In [ ]:
from harness.judge import build_judge_prompt

# exactly what a judge sees for turn 1 — note the absence of arm/model/turn
print(build_judge_prompt(sc, trace.messages[0]["content"], trace.assistant_turns[0]))

In [ ]:
# The unified constructor scripts/run_study.py uses (gap #1 closed): each judge id is
# dispatched to its SDK family, any judge that is also a subject is flagged self_judging,
# and offline=True falls back to the keyless synthetic panel — one call, same shape LIVE
# or offline.
panel = panel_from_models(PANEL, subject_models=SUBJECT, offline=not LIVE)
print("panel judge_models:", panel.judge_models, "\n")

# The SAME judging loop scripts/run_study.py uses: user turns from the message list,
# zipped with assistant turns, 1-indexed.
user_turns = [m["content"] for m in trace.messages if m["role"] == "user"]
demo_judgments = [panel.judge_turn(sc, ut, at, spec.key, i)          # -> list[TurnJudgment], one per judge
                  for i, (ut, at) in enumerate(zip(user_turns, trace.assistant_turns), start=1)]

for tj in demo_judgments[0]:      # turn 1, both judges
    print(f"[{tj.judge_model}] turn {tj.turn_index}: stance={tj.stance:+.3f} "
          f"challenge={tj.contains_challenge} present={tj.considerations_present} "
          f"warmth={tj.warmth:.2f} serves_obj={tj.serves_stated_objective}")

## Stage 6 — the judgments DataFrame

`judgment_to_row` flattens each `TurnJudgment` + its `RunSpec` into one tidy row — the
only thing `metrics.py` ever sees. The **`judge_model`** column is what the per-judge
split in Stage 7 (`compute_all_per_judge`) keys on.

In [ ]:
from harness.judge import judgment_to_row

demo_df = pd.DataFrame([judgment_to_row(tj, spec) for turn in demo_judgments for tj in turn])
print(f"{len(demo_df)} rows = {len(demo_judgments)} turns x {len(panel.judges)} judges")
demo_df[["judge_model", "turn_index", "arm", "model", "stance",
         "contains_challenge", "serves_stated_objective"]]

This single **pro** arm proves the per-turn plumbing, but it is far too thin for the
mirror-based battery: **UAT**, **channel divergence**, and **AAI** all need `pro` vs
`con` vs `nosource_*`, and the mock has no planted signal anyway. So — exactly as
`walkthrough.ipynb` does — the battery switches to the repo's planted-truth generator.

## Stage 7 — the per-judge metrics battery (the payoff)

`simulate_judgments` emits a frame with the **identical columns** the real judge
produces (`metrics.py` can't tell the difference), with a *planted* ground truth so the
metric code is validated against a signal put there on purpose. Two profiles:
`holds` (content invariant, delivery adapts, friction persists) and `drifts` (never
flips, never lies, but friction decays and the inventory attrits asymmetrically).

**Deliberately synthetic names** — `holds-profile` / `drifts-profile` — so nobody reads
a claim about the real subjects. Two independent seeds are re-tagged as the two panel
judges: a stand-in for two real cross-family judges coding the same conversations.

In [ ]:
from harness.simulate import simulate_judgments

PROFILE_MODELS = {"holds-profile": "holds", "drifts-profile": "drifts"}   # NOT the real subject ids
sc_dict = {SCENARIO_ID: sc}

parts = []
for jm, seed in zip(PANEL, (101, 202)):        # two seeds -> two independent judge draws
    dfj = simulate_judgments(sc_dict, models=PROFILE_MODELS, n_turns=12,
                             n_replicates=4, seed=seed).copy()
    dfj["judge_model"] = jm                     # re-tag: simulate stamps 'synthetic'
    parts.append(dfj)
battery = pd.concat(parts, ignore_index=True)
print("battery:", battery.shape,
      "| judges:", sorted(battery.judge_model.unique()),
      "| profiles:", sorted(battery.model.unique()),
      "| arms:", sorted(battery.arm.unique()))

In [ ]:
from harness.metrics import compute_all_per_judge, friction_half_life

# THE RULE (providers.JudgePanel docstring / START_HERE): compute every metric
# SEPARATELY per judge. compute_all groups by `model`, NOT `judge_model`, so
# concatenating judges into one call would silently POOL them. gap #2 is now CLOSED in
# the library: compute_all_per_judge enforces the split, returning
# {judge_model: compute_all(...)} — the exact dict the old hand-rolled groupby built,
# so everything downstream (per_judge.items(), out["friction"], the Stage 8 figures)
# is unchanged.
per_judge = compute_all_per_judge(battery, sc_dict)

print("Does the drift finding replicate across BOTH judges?\n")
for jm, out in per_judge.items():
    print(f"[{jm}]")
    for model_name in sorted(out["friction"].model.unique()):
        fhl = friction_half_life(out["friction"], model=model_name)
        cov = out["coverage_attrition"]
        cov = cov[(cov.model == model_name) & (cov.arm.isin(["pro", "con"]))]
        aai_last = cov[cov.turn_index == cov.turn_index.max()].aai.mean()
        print(f"  {model_name:15s} half_life={str(fhl.get('empirical_half_life')):>4s}  "
              f"terminal_friction={fhl.get('terminal_rate', float('nan')):.2f}  "
              f"final_AAI={aai_last:+.2f}")
    print()

> **Read this before reading the numbers.** The battery above is a **wire test of the
> analysis**, not a measurement of anything. The `holds` / `drifts` labels are planted,
> not observed. The two "judges" are two synthetic seeds, not real cross-family models.
> What this *does* show — and all it shows — is that the real `compute_all` + `plots`
> code recovers a planted signal, and that the per-judge split asks the right question:
> *does the finding hold up under a second, independent judge?*

## Stage 8 — figures, per judge, and the map to the real study

Every `plots.*` function saves a PNG and returns its `Path`; we embed each with
`IPython.display.Image`. Figures separate by `model`, so within each judge the
`holds-profile` and `drifts-profile` lines can be read against each other. Rendering the
family for **both** judges is the visual form of the replication check.

In [ ]:
from harness import plots
from IPython.display import Image, display


def render_family(out, tag):
    return {
        "channel_separation":   plots.fig_channel_separation(out["divergence"],          FIG_DIR / f"{tag}_channel_separation.png"),
        "friction_survival":    plots.fig_friction_survival(out["judgments_enriched"],   FIG_DIR / f"{tag}_friction_survival.png"),
        "asymmetric_attrition": plots.fig_asymmetric_attrition(out["coverage_attrition"], FIG_DIR / f"{tag}_asymmetric_attrition.png"),
        "speaker_free_floor":   plots.fig_speaker_free_floor(out["uat"],                 FIG_DIR / f"{tag}_speaker_free_floor.png"),
        "horizon":              plots.fig_horizon(out["horizon"],                        FIG_DIR / f"{tag}_horizon.png"),
        "flip_blindspot":       plots.fig_flip_blindspot(out["judgments_enriched"],      FIG_DIR / f"{tag}_flip_blindspot.png"),
    }


fam = {}
for jm, out in per_judge.items():
    fam[jm] = render_family(out, jm.replace(".", "_").replace("-", "_"))
    print(f"[{jm}] saved {len(fam[jm])} figures to {FIG_DIR}")

In [ ]:
# Headline — where does the model adapt? Content flat + delivery rising = holds.
for jm in per_judge:
    print(f"=== {jm}: content vs delivery divergence ===")
    display(Image(str(fam[jm]["channel_separation"])))

In [ ]:
# The argument for the whole battery: a flip metric sees nothing; friction survival does.
for jm in per_judge:
    print(f"=== {jm}: the flip metric's blind spot ===")
    display(Image(str(fam[jm]["flip_blindspot"])))

In [ ]:
# The rest of the family for the first judge (the second reproduces the same shape).
lead = list(per_judge)[0]
print(f"Full battery for {lead}:")
for name in ("speaker_free_floor", "asymmetric_attrition", "friction_survival", "horizon"):
    display(Image(str(fam[lead][name])))

### Map to the real study

The real run of this scenario, end to end:

```
python scripts/run_study.py --scenarios S04_tanking_strategy --replicates 2
```

**`--replicates 2` is a floor, not a preference.** The per-judge summary's denoise step
estimates the within-arm noise floor from replicates of the *same* arm, so it needs **≥2
replicates per arm**. With `--replicates 1`, `run_study.py` still completes and writes the
judgments but **skips the per-judge summary best-effort** — it has no same-arm pairs to
denoise; calling `metrics.compute_all` directly on a one-replicate frame, as this notebook
does in Stage 7, raises instead. Either way ≥2 is the minimum for a run whose numbers can
be analysed. (`compute_all`/`denoise` are left unchanged — the requirement is a
precondition, not a bug.)

What the merged judge-panel work changed — gaps #1/#2 are now **closed in the driver**,
not just here in the notebook:

- **The cross-family panel is the default.** `run_study.py` builds it via the same
  `providers.panel_from_models(...)` Stage 5 calls: `--judge-models` defaults to
  `gemini-3.1-pro gpt-5.6-terra`. Pass `--judge-model <id>` for a single-judge run, or
  `--offline` for a **no-spend rehearsal** on the synthetic panel — the `LIVE = False`
  path of this notebook, driven from the CLI.
- **The per-judge split is enforced by the library.** Each result row carries
  `judge_model`, and `metrics.compute_all_per_judge` reports every judge separately.
  Pooling judges is no longer something a caller can do by accident.
- **`--controls-only`** runs just the positive/negative controls — the cheap sanity
  pass to clear *before* committing to a full paid battery.

And the budget lever everyone forgets: **input tokens scale with the *square* of turn
count**, because every turn re-sends the whole history — and now **× the number of
judges** in the panel (see the cost cell below).

In [ ]:
from harness.runner import estimate_cost

# estimate_cost prices a SINGLE judge (one judgment per turn). The cross-family panel is
# now the default, so the judging half scales x n_judges; generation cost does not. Use
# >=2 replicates to match the run that compute_all's denoise floor actually accepts.
N_JUDGES = len(PANEL)
est = estimate_cost(n_scenarios=1, n_arms=len(ARMS), n_models=len(SUBJECT),
                    n_replicates=2, n_turns=12)
for k, v in est.items():
    print(f"  {k:26s} {v}")

panel_judging = round(est["judging_cost_usd"] * N_JUDGES, 2)
panel_total   = round(est["generation_cost_usd"] + panel_judging, 2)
print(f"\n  with the {N_JUDGES}-judge panel (judging half x n_judges):")
print(f"    {'judging_cost_usd':26s} {panel_judging}")
print(f"    {'total_cost_usd':26s} {panel_total}")
print("\n(defaults are placeholder prices — verify current per-token pricing before relying on the $ figure)")

## The one-paragraph version

One pre-registered scenario, two mirrored user scripts that differ only in which side
the user takes, run through the *real* harness with a mock subject and synthetic judges
so the whole pipeline executes with no key and no spend. A blind panel codes each turn
independently; the judgments become a tidy frame; the metric battery is computed
**separately per judge** and the figures are read for replication. The offline numbers
are a wire test, not a measurement — the planted-truth battery in Stage 7 is what
validates the metrics — but every code path here is the one a real, paid run executes,
and of the three honesty rails, two are now closed in the library — the study driver
builds the cross-family panel and `compute_all_per_judge` enforces the per-judge split —
while the third, the unverifiable model strings, is inherent and stays surfaced rather than
smoothed over. Flip `LIVE = True`, clear the
Stage 1 gate, and the same cells make real cross-family calls.